In [1]:
import os
os.chdir("/kaggle/working")
!git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...
remote: Enumerating objects: 1135, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 1135 (delta 37), reused 56 (delta 20), pack-reused 1057 (from 3)
Receiving objects: 100% (1135/1135), 434.75 MiB | 35.64 MiB/s, done.
Resolving deltas: 100% (587/587), done.
Updating files: 100% (270/270), done.
Filtering content: 100% (4/4), 795.41 MiB | 38.53 MiB/s, done.
Encountered 6 file(s) that should have been pointers, but weren't:
	ham10000/checkpoints/baseline_image_only/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/last_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/best_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/last_model.pt
	ham10000/checkpoints/metadata_fusion/best_model.pt


## SETUP PATH

In [2]:
import shutil, os

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = "/kaggle/working/project/ham10000/data"
os.makedirs(WORK, exist_ok=True)

for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    src = f"{KAGGLE_DATA}/{folder}"
    dst = f"{WORK}/{folder}"
    if not os.path.exists(src):
        raise FileNotFoundError(f"Source folder missing: {src}")
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)

for fname in ["HAM10000_split.csv", "class_weights.npy", "HAM10000_metadata.csv"]:
    src = f"{KAGGLE_DATA}/{fname}"
    dst = f"{WORK}/{fname}"
    if not os.path.exists(src):
        print(f"WARNING: {fname} not found at {src} — skipping (may need to be generated separately)")
        continue
    shutil.copy(src, dst)

print("Paths ready")

Paths ready


## TRAINING USING resnet18_v3_balanced.yaml

In [3]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# Corrected path — confirmed via os.listdir() drill-down:
# /kaggle/input -> datasets -> rehmandaket -> ham10000 -> data
KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA} — "
        "run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') to re-check the path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================
GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================
exp_name = "resnet18_v3_balanced"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
#
# Key differences from v2_tuned_stable:
#   - freeze_epochs: 0        -> train end-to-end, no staged freezing
#   - dropout: 0.3, wd: 1e-4  -> back to baseline-level regularization
#   - use_weighted_sampler: true + loss=focal (no alpha)
#       -> ONE class-imbalance mechanism (sampler), not two stacked
#          (sampler + effective_num alpha was fighting itself before)
# ============================================================
cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "focal",
        "focal_gamma": 1.5,
        # no alpha / class_counts here on purpose —
        # WeightedRandomSampler below handles class balance instead
    },

    "model": {
        "architecture": "resnet18",
        "dropout": 0.3,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "epochs": 40,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": False,
        "use_weighted_sampler": True,
        "weight_decay": 1e-4,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================
print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f'python {REPO}/ham10000/src/train.py --config {config_path}'
)

# ============================================================
# Save Results to GitHub
# ============================================================
print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your token or GitHub permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v3_balanced.yaml
TRAINING: resnet18_v3_balanced


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v3_balanced
Architecture: resnet18
Metadata    : False
EMA         : False
Freeze ep   : 0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 151MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:317: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 40 epochs — loss=focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` 

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   df45a16..8e9a7d9  main -> main


## TRAINING WITH resnet18_v3_balanced_annealed.yaml

In [6]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)
    os.symlink(src, dst)
print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================
GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)
os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================
exp_name = "resnet18_v4_balanced_annealed"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
#
# Changes from v3_balanced, and why:
#   - epochs: 25 (was 40)         -> cosine schedule now matches the
#                                    realistic training length, so LR
#                                    actually reaches near-zero by the
#                                    end instead of being cut off mid-anneal
#   - early_stopping_patience: 18 -> long enough to survive the noisy
#                                    middle epochs and reach the low-LR
#                                    consolidation phase (v3 got cut off
#                                    at epoch 22/40 with LR still at 4e-5)
#   - focal_gamma: 1.0 (was 1.5)  -> sampler is already doing heavy
#                                    rebalancing (df oversampled ~62x);
#                                    a softer gamma avoids compounding
#                                    that with extra hard-example emphasis
#                                    on the same repeated rare-class images
# ============================================================
cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "focal",
        "focal_gamma": 1.0,
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.3,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 18,
        "epochs": 25,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": False,
        "use_weighted_sampler": True,
        "weight_decay": 1e-4,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"
with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)
print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================
print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)
get_ipython().system(
    f'python {REPO}/ham10000/src/train.py --config {config_path}'
)

# ============================================================
# Save Results to GitHub
# ============================================================
print("\nSaving results to GitHub...")
os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)
os.system(f'git commit -m "Add {exp_name} training results"')
result = os.system("git push origin main")
if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your token or GitHub permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Updating 8e9a7d9..51cb50c
Fast-forward
 ...anced.yaml => resnet_v3_balanced_annealed.yaml} | 42 +++++++++++-----------
 1 file changed, 21 insertions(+), 21 deletions(-)
 rename ham10000/configs/{resnet_v3_balanced.yaml => resnet_v3_balanced_annealed.yaml} (58%)

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v4_balanced_annealed.yaml
TRAINING: resnet18_v4_balanced_annealed


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD
   8e9a7d9..51cb50c  main       -> origin/main



Device      : cuda
Experiment  : resnet18_v4_balanced_annealed
Architecture: resnet18
Metadata    : False
EMA         : False
Freeze ep   : 0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:317: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 25 epochs — loss=focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):
Epoch   1/25 | train=0.7100 | val=0.6512 | bal_acc=0.6831 | lr=9.96e-0

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   51cb50c..7f7a6e3  main -> main


## SAVE PNG FILE FOR BOTH 

In [7]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------
GITHUB_TOKEN = getpass.getpass("GitHub Token: ")
REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')
os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)
os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiments to plot
# --------------------------------------------------
exp_names = [
    "resnet18_v3_balanced",
    "resnet18_v4_balanced_annealed",
]

png_paths = []

for exp_name in exp_names:
    log_path = f"ham10000/experiments/{exp_name}/training_log.csv"
    if not os.path.exists(log_path):
        print(f"WARNING: {log_path} not found — skipping {exp_name}")
        continue

    log = pd.read_csv(log_path)

    # --------------------------------------------------
    # Create Figure
    # --------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    ax1.plot(
        log["epoch"],
        log["train_loss"],
        linewidth=2,
        linestyle="--",
        label="Train Loss"
    )
    ax1.plot(
        log["epoch"],
        log["val_loss"],
        linewidth=2,
        label="Validation Loss"
    )
    ax2.plot(
        log["epoch"],
        log["val_bal_acc"],
        linewidth=2.5,
        label="Balanced Accuracy"
    )

    best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
    best_acc = float(log["val_bal_acc"].max())
    ax2.scatter(best_epoch, best_acc, s=80)
    ax2.annotate(
        f"Best={best_acc:.4f}",
        (best_epoch, best_acc),
        xytext=(best_epoch + 1, best_acc - 0.03),
        arrowprops=dict(arrowstyle="->")
    )

    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax2.set_ylabel("Balanced Accuracy")
    plt.title(f"{exp_name}\nTraining Curves")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

    plt.tight_layout()
    png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"
    plt.savefig(png_path, dpi=300)
    plt.close()

    print("Saved:", png_path)
    png_paths.append(png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------
if png_paths:
    os.system(f"git add {' '.join(png_paths)}")
    os.system('git commit -m "Add training curves for v3_balanced and v4_balanced_annealed"')
    result = os.system("git push origin main")
    if result == 0:
        print("\nPNG(s) pushed successfully!")
    else:
        print("\nPush failed.")
else:
    print("\nNo PNGs generated — nothing to push.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v3_balanced_training_curves.png
Saved: ham10000/results/figures/resnet18_v4_balanced_annealed_training_curves.png
[main 52b592f] Add training curves for v3_balanced and v4_balanced_annealed
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v3_balanced_training_curves.png
 create mode 100644 ham10000/results/figures/resnet18_v4_balanced_annealed_training_curves.png

PNG(s) pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   7f7a6e3..52b592f  main -> main


## TRAINING WITH resnet18_v2_tuned_gamma1.yaml

In [3]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v2_tuned_gamma1"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
#
# Changes:
# - Focal gamma reduced from 2.0 → 1.0
# - Weighted focal loss using Effective Number weighting
# - Cosine LR scheduler
# - EMA enabled
# - 45 training epochs
# - Initial backbone freezing for 4 epochs
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "class_counts": [890, 5367, 412, 257, 891, 86, 114],
        "effective_num_beta": 0.999,
        "focal_gamma": 1.0,
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.4,
        "freeze_epochs": 4,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 10,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 7.94e-5,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": False,
        "weight_decay": 5e-4,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your token or GitHub permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v2_tuned_gamma1.yaml
TRAINING: resnet18_v2_tuned_gamma1


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v2_tuned_gamma1
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 4
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 144MB/s]
Backbone frozen for first 4 epoch(s).
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:317: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   ae92a4c..c4aa64b  main -> main


## SAVE PNG FILE

In [4]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------

GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiment to plot
# --------------------------------------------------

exp_name = "resnet18_v2_tuned_gamma1"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.tight_layout()

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------

os.system(f"git add {png_path}")

os.system(
    'git commit -m "Add training curves for resnet18_v2_tuned_gamma1"'
)

result = os.system("git push origin main")

if result == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v2_tuned_gamma1_training_curves.png
[main e0f33bc] Add training curves for resnet18_v2_tuned_gamma1
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v2_tuned_gamma1_training_curves.png

PNG pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   c4aa64b..e0f33bc  main -> main


## TRAINING WITH resnet18_v2_tuned_stable_unfreeze1.yaml

In [ ]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v2_tuned_stable_unfreeze1"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.999,
        "focal_gamma": 1.5,
        "class_counts": [890, 5367, 412, 257, 891, 86, 114],
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 1,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": False,
        "use_weighted_sampler": False,
        "weight_decay": 2e-4,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v2_tuned_stable_unfreeze1.yaml
TRAINING: resnet18_v2_tuned_stable_unfreeze1


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v2_tuned_stable_unfreeze1
Architecture: resnet18
Metadata    : False
EMA         : False
Freeze ep   : 1
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 166MB/s]
Backbone frozen for first 1 epoch(s).
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:317: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please

## TRAINING WITH resnet18_v6_ema_stable.yaml

In [3]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v6_ema_stable"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "class_counts": [890, 5367, 412, 257, 891, 86, 114],
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.4,
        "freeze_epochs": 3,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 15,
        "ema_decay": 0.998,
        "epochs": 50,
        "gradient_accumulation": 2,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 4e-4,
        "best_metric_smoothing_window": 3,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v6_ema_stable.yaml
TRAINING: resnet18_v6_ema_stable


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v6_ema_stable
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 3
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 238MB/s]
Backbone frozen for first 3 epoch(s).
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:340: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 50 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:250: Fu

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   275a22d..99fee20  main -> main


## SAVE PNG FILE 

In [3]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------

GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiment to plot
# --------------------------------------------------

exp_name = "resnet18_v6_ema_stable"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.tight_layout()

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------

os.system(f"git add {png_path}")

os.system(
    'git commit -m "Add training curves for resnet18_v6_ema_stable"'
)

result = os.system("git push origin main")

if result == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v6_ema_stable_training_curves.png
[main 1eafdfb] Add training curves for resnet18_v6_ema_stable
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v6_ema_stable_training_curves.png

PNG pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   99fee20..1eafdfb  main -> main


## TRAINING USING resnet18_v6_ema_sampler.yaml

In [6]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v6_ema_sampler"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "class_counts": [890, 5367, 412, 257, 891, 86, 114],
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v6_ema_sampler.yaml
TRAINING: resnet18_v6_ema_sampler


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v6_ema_sampler
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 172MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Checkpoint Directory : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/resnet18_v6_ema_sampler
Log Directory        : /kaggle/working/Trust

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   88e54b0..5ed7a7d  main -> main


## TRAINING CURVE 

In [2]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------

GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiment to plot
# --------------------------------------------------

exp_name = "resnet18_v6_ema_sampler"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.tight_layout()

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------

os.system(f"git add {png_path}")

os.system(
    'git commit -m "Add training curves for resnet18_v6_ema_sampler"'
)

result = os.system("git push origin main")

if result == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v6_ema_sampler_training_curves.png
[main f3619e6] Add training curves for resnet18_v6_ema_sampler
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v6_ema_sampler_training_curves.png

PNG pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   ba7bd44..f3619e6  main -> main


## TRAINING USING resnet18_v7_noOverfit.yaml

In [1]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v7_noOverfit"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 3,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 10,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 2,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "sampler_power": 0.5,
        "mixup_alpha": 0.3,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train Model
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

commit_status = os.system(f'git commit -m "Add {exp_name} training results"')

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v7_noOverfit.yaml
TRAINING: resnet18_v7_noOverfit


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v7_noOverfit
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 3
Mixup alpha : 0.3
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 140MB/s]
Backbone frozen for first 3 epoch(s).
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:375: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   a5a8534..d53d791  main -> main


## TRAINING CURVE

In [2]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------

GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiment to Plot
# --------------------------------------------------

exp_name = "resnet18_v9_sampler_restore"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Training Loss
ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

# Validation Loss
ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

# Balanced Accuracy
ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

# Best Epoch
best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.tight_layout()

# --------------------------------------------------
# Save Figure
# --------------------------------------------------

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v9_sampler_restore_training_curves.png
[main a912682] Add training curves for resnet18_v9_sampler_restore
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v9_sampler_restore_training_curves.png

Push failed.


fatal: could not read Password for 'https://g@github.com': No such device or address


## TRAINING USING resnet18_v8_balanced.yaml

In [1]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v8_balanced"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.97,
        "focal_gamma": 1.5,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.30,
        "freeze_epochs": 3,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 10,
        "ema_decay": 0.999,
        "epochs": 50,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "sampler_power": 0.85,
        "mixup_alpha": 0.1,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train Model
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add "
    f"ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

commit_status = os.system(
    f'git commit -m "Add {exp_name} training results"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v8_balanced.yaml
TRAINING: resnet18_v8_balanced


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v8_balanced
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 3
Mixup alpha : 0.1
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 142MB/s]
Backbone frozen for first 3 epoch(s).
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:375: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 50 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   53dc5d0..d71e80f  main -> main


## TRAINING CURVE 

In [4]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------

GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiment to Plot
# --------------------------------------------------

exp_name = "resnet18_v8_balanced"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Training Loss
ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

# Validation Loss
ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

# Balanced Accuracy
ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

# Best Epoch
best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.tight_layout()

# --------------------------------------------------
# Save Figure
# --------------------------------------------------

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v8_balanced_training_curves.png
[main fe9f95a] Add training curves for resnet18_v8_balanced
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v8_balanced_training_curves.png

PNG pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   d71e80f..fe9f95a  main -> main


## TRAINING USING resnet18_v9_tuned.yaml

In [1]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v9_tuned"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.999,
        "focal_gamma": 1.5,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": False,
        "weight_decay": 2e-4,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train Model
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add "
    f"ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

commit_status = os.system(
    f'git commit -m "Add {exp_name} training results"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v9_tuned.yaml
TRAINING: resnet18_v9_tuned


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v9_tuned
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 213MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:375: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args.

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   39620cf..6a96ed9  main -> main


## TRAINING CURVE 

In [4]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------

GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiment to Plot
# --------------------------------------------------

exp_name = "resnet18_v9_tuned"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Training Loss
ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

# Validation Loss
ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

# Balanced Accuracy
ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

# Best Epoch
best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.tight_layout()

# --------------------------------------------------
# Save Figure
# --------------------------------------------------

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v9_tuned_training_curves.png
[main 388d380] Add training curves for resnet18_v9_tuned
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v9_tuned_training_curves.png

PNG pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   a912682..388d380  main -> main


## TRAINING USING resnet18_v10.yaml

In [5]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(f"KAGGLE_DATA not found at {KAGGLE_DATA}")

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v10"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,
    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },
    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },
    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.999,
        "focal_gamma": 1.5,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },
    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },
    "output": {
        "checkpoint_dir": ckpt_dir,
    },
    "train": {
        "batch_size": 32,
        "best_metric_smoothing_window": 5,
        "early_stopping_patience": 20,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": False,
        "weight_decay": 2e-4,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train Model
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add "
    f"ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

commit_status = os.system(
    f'git commit -m "Add {exp_name} training results"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v10.yaml
TRAINING: resnet18_v10


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v10
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 161MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:375: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` 

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   dfa0189..9c4e236  main -> main


## TRAINING CURVE

In [6]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------
# Git Authentication
# --------------------------------------------------

GITHUB_TOKEN = getpass.getpass("GitHub Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# --------------------------------------------------
# Experiment to Plot
# --------------------------------------------------

exp_name = "resnet18_v10"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# --------------------------------------------------
# Create Figure
# --------------------------------------------------

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Training Loss
ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss"
)

# Validation Loss
ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

# Balanced Accuracy
ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy"
)

# Best Epoch
best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best={best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->")
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.tight_layout()

# --------------------------------------------------
# Save Figure
# --------------------------------------------------

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300)
plt.close()

print("Saved:", png_path)

# --------------------------------------------------
# Push to GitHub
# --------------------------------------------------

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nPNG pushed successfully!")
else:
    print("\nPush failed.")

GitHub Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved: ham10000/results/figures/resnet18_v10_training_curves.png
[main 82c6d77] Add training curves for resnet18_v10
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v10_training_curves.png

PNG pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   9c4e236..82c6d77  main -> main


## TRAINING USING resnet18_v11_wd.yaml

In [3]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# Kaggle Dataset Path
KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"

WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}\n"
        "Check the dataset path using os.listdir()."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v11_wd"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 5e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train Model
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your token or GitHub permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v11_wd.yaml
TRAINING: resnet18_v11_wd


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v11_wd
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 189MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:289: FutureWarning: `torch.cuda.am

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   268a3ce..d8d42de  main -> main


## TRAINING CURVE 

In [4]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "resnet18_v11_wd"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Training Loss
ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# Validation Loss
ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# Balanced Accuracy
ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# Highlight Best Epoch
best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your token or GitHub permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/resnet18_v11_wd_training_curves.png
[main fe78a27] Add training curves for resnet18_v11_wd
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v11_wd_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   d8d42de..fe78a27  main -> main


## TRAINING using resnet18_v12_smooth.yaml

In [5]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# Kaggle Dataset Path
KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"

WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}\n"
        "Check the dataset path using os.listdir()."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v12_smooth"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train Model
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your token or GitHub permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v12_smooth.yaml
TRAINING: resnet18_v12_smooth


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v12_smooth
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 176MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:289: FutureWarning: `torch.cud

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   fe78a27..a2c865b  main -> main


## TRAINING CURVE

In [6]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "resnet18_v12_smooth"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Training Loss
ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# Validation Loss
ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# Balanced Accuracy
ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# Highlight Best Epoch
best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your token or GitHub permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/resnet18_v12_smooth_training_curves.png
[main 66f2409] Add training curves for resnet18_v12_smooth
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v12_smooth_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   a2c865b..66f2409  main -> main


## TRAINING USING resnet18_v13_disclr.yaml

In [1]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# Kaggle Dataset Path

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"

WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}\n"
        "Check the dataset path using os.listdir()."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v13_disclr"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "backbone_lr_ratio": 0.1,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train Model
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your token or GitHub permissions.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v13_disclr.yaml
TRAINING: resnet18_v13_disclr


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v13_disclr
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 163MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:289: FutureWarning: `torch.cud

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   66f2409..f718ef2  main -> main


## TRAINING CURVE

In [2]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "resnet18_v13_disclr"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Training Loss
ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# Validation Loss
ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# Balanced Accuracy
ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# Highlight Best Epoch
best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your token or GitHub permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/resnet18_v13_disclr_training_curves.png
[main 3380047] Add training curves for resnet18_v13_disclr
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v13_disclr_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   f718ef2..3380047  main -> main


## TRAINING USING resnet18_v14_wd_smooth.yaml


In [3]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# Correct dataset path
KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet18_v14_wd_smooth"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "resnet18",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 5e-4,
        "best_metric_smoothing_window": 5,
        # backbone_lr_ratio intentionally omitted
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your GitHub token or repository permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet18_v14_wd_smooth.yaml
TRAINING: resnet18_v14_wd_smooth


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet18_v14_wd_smooth
Architecture: resnet18
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 143MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:289: FutureWarning: `torch.

Everything up-to-date


## TRAINING CURVE

In [4]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "resnet18_v14_wd_smooth"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# ------------------------------------------------------------
# Training Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# ------------------------------------------------------------
# Validation Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# ------------------------------------------------------------
# Balanced Accuracy
# ------------------------------------------------------------

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# ------------------------------------------------------------
# Highlight Best Epoch
# ------------------------------------------------------------

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80)

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your token or GitHub permissions.")
    

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/resnet18_v14_wd_smooth_training_curves.png
[main 6079de2] Add training curves for resnet18_v14_wd_smooth
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet18_v14_wd_smooth_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   9b0c846..6079de2  main -> main


## TRAINING USING resnet50_v12recipe.yaml

In [7]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Correct dataset path
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet50_v12recipe"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "resnet50",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your GitHub token or repository permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet50_v12recipe.yaml
TRAINING: resnet50_v12recipe


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet50_v12recipe
Architecture: resnet50
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 152MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Checkpoint Directory : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/resnet50_v12recipe
Log Directory        : /kaggle/working/Trustworthy-AI-

Everything up-to-date


## TRAINING CURVE

In [2]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "resnet50_v12recipe"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# ------------------------------------------------------------
# Training Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# ------------------------------------------------------------
# Validation Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# ------------------------------------------------------------
# Balanced Accuracy
# ------------------------------------------------------------

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# ------------------------------------------------------------
# Highlight Best Epoch
# ------------------------------------------------------------

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80, color="red")

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/resnet50_v12recipe_training_curves.png
[main eed0556] Add training curves for resnet50_v12recipe
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet50_v12recipe_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   6db120b..eed0556  main -> main


## TRAINING USING efficientnetb0_v12rceipe.yaml

In [10]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Correct dataset path
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "efficientnet_b0_v12recipe"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "efficientnet_b0",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Verify Results
# ============================================================

print("\nChecking generated files...")

best_ckpt = f"ham10000/checkpoints/{exp_name}/best_model.pt"
last_ckpt = f"ham10000/checkpoints/{exp_name}/last_model.pt"
log_file = f"ham10000/experiments/{exp_name}/training_log.csv"

print("\nCheckpoint Status")
print("-" * 50)

for f in [best_ckpt, last_ckpt, log_file, config_path]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f"✓ {f} ({size:.2f} MB)")
    else:
        print(f"✗ Missing: {f}")

# Stop immediately if checkpoints were not created
if not os.path.exists(best_ckpt):
    raise FileNotFoundError(
        f"\nERROR: {best_ckpt} was not created.\n"
        "Training completed but no best checkpoint exists."
    )

if not os.path.exists(last_ckpt):
    raise FileNotFoundError(
        f"\nERROR: {last_ckpt} was not created.\n"
        "Training completed but no last checkpoint exists."
    )

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nAdding files to Git...")

os.system(f'git add "{config_path}"')
os.system(f'git add "ham10000/checkpoints/{exp_name}"')
os.system(f'git add "ham10000/experiments/{exp_name}"')

print("\nGit Status")
print("=" * 60)
os.system("git status")
print("=" * 60)

commit_msg = f"Add {exp_name} checkpoints and training results"

commit_status = os.system(
    f'git commit -m "{commit_msg}"'
)

if commit_status != 0:
    print("\nNothing new to commit.")
else:
    print("\nCommit successful.")

print("\nPushing to GitHub...")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\n======================================")
    print("SUCCESS!")
    print("======================================")
    print(f"Experiment : {exp_name}")
    print(f"Best Model : {best_ckpt}")
    print(f"Last Model : {last_ckpt}")
    print(f"Training Log : {log_file}")
    print("Everything has been pushed to GitHub.")
else:
    raise RuntimeError(
        "Git push failed. Check your GitHub token or repository permissions."
    )

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/efficientnet_b0_v12recipe.yaml
TRAINING: efficientnet_b0_v12recipe


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : efficientnet_b0_v12recipe
Architecture: efficientnet_b0
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 140MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Checkpoint Directory : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/efficientnet_b0_v12recipe

Everything up-to-date


## TRAINING CURVE 

In [11]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "efficientnet_b0_v12recipe"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# ------------------------------------------------------------
# Training Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# ------------------------------------------------------------
# Validation Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# ------------------------------------------------------------
# Balanced Accuracy
# ------------------------------------------------------------

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# ------------------------------------------------------------
# Highlight Best Epoch
# ------------------------------------------------------------

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80, color="red")

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/efficientnet_b0_v12recipe_training_curves.png
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Nothing new to commit or commit failed.

Training curve pushed successfully!


Everything up-to-date


## TRAINING USING efficientnetb1_v12recipe.yaml

In [5]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Correct dataset path
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "efficientnet_b1_v12recipe"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "efficientnet_b1",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your GitHub token or repository permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Updating 43bf073..7d977a6
Fast-forward
 ham10000/configs/densenet121_v12recipe.yaml | 47 +++++++++++++++++++++++++++++
 ham10000/configs/resnet101_v12recipe.yaml   | 47 +++++++++++++++++++++++++++++
 2 files changed, 94 insertions(+)
 create mode 100644 ham10000/configs/densenet121_v12recipe.yaml
 create mode 100644 ham10000/configs/resnet101_v12recipe.yaml

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/efficientnet_b1_v12recipe.yaml
TRAINING: efficientnet_b1_v12recipe


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD
   43bf073..7d977a6  main       -> origin/main



Device      : cuda
Experiment  : efficientnet_b1_v12recipe
Architecture: efficientnet_b1
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth
100%|███████████████████████████████████████| 30.1M/30.1M [00:00<00:00, 225MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   7d977a6..d729844  main -> main


## TRAINING CURVE

In [6]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "efficientnet_b1_v12recipe"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# ------------------------------------------------------------
# Training Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# ------------------------------------------------------------
# Validation Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# ------------------------------------------------------------
# Balanced Accuracy
# ------------------------------------------------------------

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# ------------------------------------------------------------
# Highlight Best Epoch
# ------------------------------------------------------------

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80, color="red")

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/efficientnet_b1_v12recipe_training_curves.png
[main d928197] Add training curves for efficientnet_b1_v12recipe
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/efficientnet_b1_v12recipe_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   d729844..d928197  main -> main


## TRAINING USING densenet121_v12recipe.yaml

In [1]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Correct dataset path
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "densenet121_v12recipe"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "densenet121",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Verify Results
# ============================================================

print("\nChecking generated files...")

best_ckpt = f"ham10000/checkpoints/{exp_name}/best_model.pt"
last_ckpt = f"ham10000/checkpoints/{exp_name}/last_model.pt"
log_file = f"ham10000/experiments/{exp_name}/training_log.csv"

print("\nCheckpoint Status")
print("-" * 50)

for f in [best_ckpt, last_ckpt, log_file, config_path]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f"✓ {f} ({size:.2f} MB)")
    else:
        print(f"✗ Missing: {f}")

# Stop immediately if checkpoints were not created
if not os.path.exists(best_ckpt):
    raise FileNotFoundError(
        f"\nERROR: {best_ckpt} was not created.\n"
        "Training completed but no best checkpoint exists."
    )

if not os.path.exists(last_ckpt):
    raise FileNotFoundError(
        f"\nERROR: {last_ckpt} was not created.\n"
        "Training completed but no last checkpoint exists."
    )

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nAdding files to Git...")

os.system(f'git add "{config_path}"')
os.system(f'git add "ham10000/checkpoints/{exp_name}"')
os.system(f'git add "ham10000/experiments/{exp_name}"')

print("\nGit Status")
print("=" * 60)
os.system("git status")
print("=" * 60)

commit_msg = f"Add {exp_name} checkpoints and training results"

commit_status = os.system(
    f'git commit -m "{commit_msg}"'
)

if commit_status != 0:
    print("\nNothing new to commit.")
else:
    print("\nCommit successful.")

print("\nPushing to GitHub...")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\n======================================")
    print("SUCCESS!")
    print("======================================")
    print(f"Experiment : {exp_name}")
    print(f"Best Model : {best_ckpt}")
    print(f"Last Model : {last_ckpt}")
    print(f"Training Log : {log_file}")
    print("Everything has been pushed to GitHub.")
else:
    raise RuntimeError(
        "Git push failed. Check your GitHub token or repository permissions."
    )

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/densenet121_v12recipe.yaml
TRAINING: densenet121_v12recipe


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : densenet121_v12recipe
Architecture: densenet121
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|███████████████████████████████████████| 30.8M/30.8M [00:00<00:00, 171MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Checkpoint Directory : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/densenet121_v12recipe
Log Directory        : /kaggle/working/

Everything up-to-date


## TRAINING CURVE

In [2]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

# Configure Git
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

# Set remote URL
os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

# Pull latest changes
os.system("git pull origin main")

# Create output directory
os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "densenet121_v12recipe"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# ------------------------------------------------------------
# Training Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# ------------------------------------------------------------
# Validation Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# ------------------------------------------------------------
# Balanced Accuracy
# ------------------------------------------------------------

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# ------------------------------------------------------------
# Highlight Best Epoch
# ------------------------------------------------------------

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80, color="red")

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")
    

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/densenet121_v12recipe_training_curves.png
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Nothing new to commit or commit failed.

Training curve pushed successfully!


Everything up-to-date


## TRAINING USING resnet101_v12recipe.yaml

In [9]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Correct dataset path
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "resnet101_v12recipe"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "resnet101",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 3e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Train
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

os.system(f'git commit -m "Add {exp_name} training results"')

result = os.system("git push origin main")

if result == 0:
    print("Results pushed successfully!")
else:
    print("Git push failed. Check your GitHub token or repository permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet101_v12recipe.yaml
TRAINING: resnet101_v12recipe


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : resnet101_v12recipe
Architecture: resnet101
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth
100%|█████████████████████████████████████████| 171M/171M [00:00<00:00, 209MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:289: FutureWarning: `torch.

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   6edb518..cd68ed8  main -> main


## TRAINING CURVE 

In [10]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

# Configure Git
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

# Set remote URL
os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

# Pull latest changes
os.system("git pull origin main")

# Create output directory
os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "resnet101_v12recipe"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# ------------------------------------------------------------
# Training Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# ------------------------------------------------------------
# Validation Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# ------------------------------------------------------------
# Balanced Accuracy
# ------------------------------------------------------------

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# ------------------------------------------------------------
# Highlight Best Epoch
# ------------------------------------------------------------

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80, color="red")

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/resnet101_v12recipe_training_curves.png
[main 917d005] Add training curves for resnet101_v12recipe
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/resnet101_v12recipe_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   cd68ed8..917d005  main -> main


## TRAINING USING densenet121_v12recipe_wd.yaml

In [3]:
import os
import yaml
import getpass
import shutil

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Dataset Path
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"
CONFIG_DIR = f"{REPO}/ham10000/configs"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Experiment Name
# ============================================================

exp_name = "densenet121_v12recipe_wd"

ckpt_dir = f"ham10000/checkpoints/{exp_name}"
exp_dir = f"experiments/{exp_name}"

os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(exp_dir, exist_ok=True)

# ============================================================
# Create YAML Config
# ============================================================

cfg = {
    "seed": 42,

    "data": {
        "augment": True,
        "data_dir": WORK,
        "img_size": 224,
        "num_workers": 2,
    },

    "logging": {
        "checkpoint_dir": ckpt_dir,
        "experiment_name": exp_name,
        "save_dir": exp_dir,
    },

    "loss": {
        "name": "weighted_focal",
        "alpha_mode": "effective_num",
        "effective_num_beta": 0.99,
        "focal_gamma": 1.5,
        "label_smoothing": 0.1,
        "class_counts": [
            890,
            5367,
            412,
            257,
            891,
            86,
            114,
        ],
    },

    "model": {
        "architecture": "densenet121",
        "dropout": 0.35,
        "freeze_epochs": 0,
        "metadata_dim": 0,
        "num_classes": 7,
        "pretrained": True,
    },

    "output": {
        "checkpoint_dir": ckpt_dir,
    },

    "train": {
        "batch_size": 32,
        "early_stopping_patience": 12,
        "ema_decay": 0.999,
        "epochs": 45,
        "gradient_accumulation": 1,
        "lr": 1e-4,
        "lr_warmup_epochs": 3,
        "max_grad_norm": 1.0,
        "scheduler": "cosine",
        "use_ema": True,
        "use_weighted_sampler": True,
        "weight_decay": 5e-4,
        "best_metric_smoothing_window": 5,
    },
}

config_path = f"{CONFIG_DIR}/{exp_name}.yaml"

with open(config_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nCreated config:\n{config_path}")

# ============================================================
# Display Config
# ============================================================

print("\nGenerated YAML Configuration:")
with open(config_path, "r") as f:
    print(f.read())

# ============================================================
# Start Training
# ============================================================

print("=" * 60)
print(f"TRAINING: {exp_name}")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/train.py --config {config_path}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system(
    f"git add "
    f"ham10000/checkpoints/{exp_name} "
    f"ham10000/experiments/{exp_name} "
    f"ham10000/configs/{exp_name}.yaml"
)

commit_status = os.system(
    f'git commit -m "Add {exp_name} training results"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nResults pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.

Created config:
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/densenet121_v12recipe_wd.yaml

Generated YAML Configuration:
seed: 42
data:
  augment: true
  data_dir: /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/data
  img_size: 224
  num_workers: 2
logging:
  checkpoint_dir: ham10000/checkpoints/densenet121_v12recipe_wd
  experiment_name: densenet121_v12recipe_wd
  save_dir: experiments/densenet121_v12recipe_wd
loss:
  name: weighted_focal
  alpha_mode: effective_num
  effective_num_beta: 0.99
  focal_gamma: 1.5
  label_smoothing: 0.1
  class_counts:
  - 890
  - 5367
  - 412
  - 257
  - 891
  - 86
  - 114
model:
  architecture: densenet121
  dropout: 0.35
  freeze_epochs: 0
  metadata_dim: 0
  num_classes: 7
  pretrained: true
output:
  checkpoint_dir: ham10000/checkpoints/densenet121_v12recipe_wd
train:
  batch_size: 32
  early_stopping_patience: 12
  ema_decay: 0.999
  epochs: 45
  gradient_accumulation: 1
  lr: 0.0001

From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Device      : cuda
Experiment  : densenet121_v12recipe_wd
Architecture: densenet121
Metadata    : False
EMA         : True
Freeze ep   : 0
Mixup alpha : 0.0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
WeightedRandomSampler enabled (class counts: [890, 5367, 412, 257, 891, 86, 114])
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|███████████████████████████████████████| 30.8M/30.8M [00:00<00:00, 157MB/s]
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:385: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 45 epochs — loss=weighted_focal, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:289: FutureWarni

To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   c85b887..bb8a204  main -> main


## TRAINING CURVE 

In [4]:
import os
import getpass
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
os.chdir(REPO)

# Configure Git
os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

# Set remote URL
os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

# Pull latest changes
os.system("git pull origin main")

# Create output directory
os.makedirs("ham10000/results/figures", exist_ok=True)

# ============================================================
# Experiment to Plot
# ============================================================

exp_name = "densenet121_v12recipe_wd"

log_path = f"ham10000/experiments/{exp_name}/training_log.csv"

if not os.path.exists(log_path):
    raise FileNotFoundError(f"{log_path} not found.")

log = pd.read_csv(log_path)

# ============================================================
# Create Figure
# ============================================================

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# ------------------------------------------------------------
# Training Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["train_loss"],
    linewidth=2,
    linestyle="--",
    label="Train Loss",
)

# ------------------------------------------------------------
# Validation Loss
# ------------------------------------------------------------

ax1.plot(
    log["epoch"],
    log["val_loss"],
    linewidth=2,
    label="Validation Loss",
)

# ------------------------------------------------------------
# Balanced Accuracy
# ------------------------------------------------------------

ax2.plot(
    log["epoch"],
    log["val_bal_acc"],
    linewidth=2.5,
    label="Balanced Accuracy",
)

# ------------------------------------------------------------
# Highlight Best Epoch
# ------------------------------------------------------------

best_epoch = int(log.loc[log["val_bal_acc"].idxmax(), "epoch"])
best_acc = float(log["val_bal_acc"].max())

ax2.scatter(best_epoch, best_acc, s=80, color="red")

ax2.annotate(
    f"Best = {best_acc:.4f}",
    (best_epoch, best_acc),
    xytext=(best_epoch + 1, best_acc - 0.03),
    arrowprops=dict(arrowstyle="->"),
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax2.set_ylabel("Balanced Accuracy")

plt.title(f"{exp_name}\nTraining Curves")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right",
)

plt.tight_layout()

# ============================================================
# Save Figure
# ============================================================

png_path = f"ham10000/results/figures/{exp_name}_training_curves.png"

plt.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(f"Saved figure: {png_path}")

# ============================================================
# Push to GitHub
# ============================================================

os.system(f"git add {png_path}")

commit_status = os.system(
    f'git commit -m "Add training curves for {exp_name}"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nTraining curve pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")

GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.
Saved figure: ham10000/results/figures/densenet121_v12recipe_wd_training_curves.png
[main cfd7c6c] Add training curves for densenet121_v12recipe_wd
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ham10000/results/figures/densenet121_v12recipe_wd_training_curves.png

Training curve pushed successfully!


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   bb8a204..cfd7c6c  main -> main


## TRAINING USING ENSEMBLE EVAL: efficientnet_b0_v12recipe + densenet121_v12recipe

In [4]:
import os
import getpass

# ============================================================
# Setup: Repository + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Dataset Path
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}.\n"
        "Run os.listdir('/kaggle/input/datasets/rehmandaket/ham10000') "
        "to verify the dataset path."
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

import shutil
for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)
    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication + Pull (this is where the checkpoints
# you pushed from your laptop actually arrive on Kaggle)
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
)

os.system("git pull origin main")

# ============================================================
# Confirm the checkpoints actually pulled down
# ============================================================

eff_ckpt  = f"{REPO}/ham10000/checkpoints/efficientnet_b0_v12recipe/best_model.pt"
dense_ckpt = f"{REPO}/ham10000/checkpoints/densenet121_v12recipe/best_model.pt"

for p in [eff_ckpt, dense_ckpt]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"{p} not found after git pull — the push from your laptop "
            "may not have completed. Check GitHub for the file before rerunning."
        )
    print(f"Found {p} ({os.path.getsize(p)/1e6:.1f} MB)")

# ============================================================
# Run the ensemble evaluator
# ============================================================

print("=" * 60)
print("ENSEMBLE EVAL: efficientnet_b0_v12recipe + densenet121_v12recipe")
print("=" * 60)

get_ipython().system(
    f"python {REPO}/ham10000/src/evaluate_ensemble.py "
    f"--configs {REPO}/ham10000/configs/efficientnet_b0_v12recipe.yaml {REPO}/ham10000/configs/densenet121_v12recipe.yaml "
    f"--checkpoints {eff_ckpt} {dense_ckpt}"
)

# ============================================================
# Save Results to GitHub
# ============================================================

print("\nSaving results to GitHub...")

os.system("git add ham10000/results/")

commit_status = os.system(
    'git commit -m "Add efficientnet_b0_v12recipe + densenet121_v12recipe ensemble eval results"'
)

if commit_status != 0:
    print("Nothing new to commit or commit failed.")

push_status = os.system("git push origin main")

if push_status == 0:
    print("\nResults pushed successfully!")
else:
    print("\nGit push failed. Check your GitHub token or repository permissions.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD
   047d80a..95593b3  main       -> origin/main


Updating 047d80a..95593b3
Fast-forward
 ham10000/src/train.py | 8 ++++++++
 1 file changed, 8 insertions(+)
Found /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/efficientnet_b0_v12recipe/best_model.pt (48.7 MB)
Found /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/densenet121_v12recipe/best_model.pt (84.5 MB)
ENSEMBLE EVAL: efficientnet_b0_v12recipe + densenet121_v12recipe
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 141MB/s]
[efficientnet_b0_v12recipe] loaded epoch 16, val_bal_acc=0.8113
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|███████████████████████████████████████| 30.8M/30.8M [00:00<00:00, 160MB/s]
[densenet121_v12recipe] loaded epoc

Everything up-to-date


## Evaluate test

In [2]:
import os
import getpass
import shutil

# ============================================================
# Setup: repo + dataset linking (safe to re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Dataset Linking
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}"
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication (Optional)
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment
# ============================================================

exp_name = "densenet121_v12recipe"

config_path = f"{REPO}/ham10000/configs/{exp_name}.yaml"
checkpoint_path = f"{REPO}/ham10000/checkpoints/{exp_name}/best_model.pt"

print("=" * 60)
print(f"RUNNING TTA EVALUATION: {exp_name}")
print("=" * 60)

# ============================================================
# Evaluate with Test-Time Augmentation (TTA)
# ============================================================

get_ipython().system(
    f"""
python {REPO}/ham10000/src/evaluate_tta.py \
    --config {config_path} \
    --checkpoint {checkpoint_path} \
    --tta_passes 8
"""
)

print("\nTTA evaluation completed.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...
Updating files: 100% (262/262), done.
Filtering content: 100% (4/4), 795.41 MiB | 27.48 MiB/s, done.
Encountered 6 file(s) that should have been pointers, but weren't:
	ham10000/checkpoints/baseline_image_only/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/last_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/best_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/last_model.pt
	ham10000/checkpoints/metadata_fusion/best_model.pt


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.
RUNNING TTA EVALUATION: densenet121_v12recipe


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Experiment  : densenet121_v12recipe
Checkpoint  : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/densenet121_v12recipe/best_model.pt
TTA passes  : 8
Batch size  : 32
Metadata    : False
Device      : cuda

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|███████████████████████████████████████| 30.8M/30.8M [00:00<00:00, 169MB/s]
Loaded epoch 19, val_bal_acc=0.8148

[test] 989 images for TTA evaluation
  TTA pass 8/8...
TTA RESULTS (8 passes) — densenet121_v12recipe
Balanced Accuracy : 0.6750
Macro F1          : 0.6548
Macro ROC-AUC     : 0.9580

Per-class F1:
  mel   : 0.5868  ███████████
  nv    : 0.9031  ██████████████████
  bcc   : 0.6863  █████████████
  akiec : 0.6582  █████████████
  bkl   : 0.6460  ████████████
  df    : 0.3529  ███████
  vasc  : 0.7500  ███████████████

              precision    recall  f1-score   support

         mel     0.5462    0.63

In [3]:
import os
import getpass
import shutil

# ============================================================
# Setup: Repo + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Dataset Linking
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}"
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication (Optional)
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment
# ============================================================

exp_name = "densenet121_v12recipe"

config_path = f"{REPO}/ham10000/configs/{exp_name}.yaml"
checkpoint_path = f"{REPO}/ham10000/checkpoints/{exp_name}/best_model.pt"

print("=" * 60)
print(f"RUNNING TEST EVALUATION: {exp_name}")
print("=" * 60)

# ============================================================
# Evaluate on Test Set
# ============================================================

get_ipython().system(
    f"""
python {REPO}/ham10000/src/evaluate_test.py \
    --config {config_path} \
    --checkpoint {checkpoint_path} \
    --split test
"""
)

print("\nTest evaluation completed.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.
RUNNING TEST EVALUATION: densenet121_v12recipe


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Experiment  : densenet121_v12recipe
Checkpoint  : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/densenet121_v12recipe/best_model.pt
Split       : test
Metadata    : False
Device      : cuda

[test] 989 images | mode=test
Loaded 989 images for split='test'
Checkpoint loaded  (epoch 19, val_bal_acc=0.8148)

Running inference...

RESULTS  —  densenet121_v12recipe  —  split=test
Balanced Accuracy : 0.7039
Macro F1          : 0.6587
Macro ROC-AUC     : 0.9507

Per-class F1:
  mel   : 0.5204  ██████████
  nv    : 0.8857  █████████████████
  bcc   : 0.7059  ██████████████
  akiec : 0.6279  ████████████
  bkl   : 0.6267  ████████████
  df    : 0.4444  ████████
  vasc  : 0.8000  ████████████████

Confusion Matrix:
[[ 70  25   3   5   9   0   0]
 [ 66 554   9   3  14   4   5]
 [  0   0  36   3   4   0   1]
 [  1   0   3  27   9   1   0]
 [ 19  12   7   7  68   0   0]
 [  0   5   0   0   0   4   0]
 [  1   0   0   0   0   0  14]]

Classification Report:
            

In [4]:
import os
import getpass
import shutil

# ============================================================
# Setup: Repo + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Dataset Linking
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}"
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

os.system("git pull origin main")

# ============================================================
# Experiment
# ============================================================

exp_name = "resnet18_v6_ema_sampler"

config_path = f"{REPO}/ham10000/configs/{exp_name}.yaml"
checkpoint_path = f"{REPO}/ham10000/checkpoints/{exp_name}/best_model.pt"

print("=" * 60)
print(f"RUNNING TEST EVALUATION: {exp_name}")
print("=" * 60)

# ============================================================
# Evaluate on Test Set
# ============================================================

get_ipython().system(
    f"""
python {REPO}/ham10000/src/evaluate_test.py \
    --config {config_path} \
    --checkpoint {checkpoint_path} \
    --split test
"""
)

print("\nTest evaluation completed.")

Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········


Already up to date.
RUNNING TEST EVALUATION: resnet18_v6_ema_sampler


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD



Experiment  : resnet18_v6_ema_sampler
Checkpoint  : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/resnet18_v6_ema_sampler/best_model.pt
Split       : test
Metadata    : False
Device      : cuda

[test] 989 images | mode=test
Loaded 989 images for split='test'
Checkpoint loaded  (epoch 25, val_bal_acc=0.8075)

Running inference...

RESULTS  —  resnet18_v6_ema_sampler  —  split=test
Balanced Accuracy : 0.7064
Macro F1          : 0.6838
Macro ROC-AUC     : 0.9522

Per-class F1:
  mel   : 0.4735  █████████
  nv    : 0.8693  █████████████████
  bcc   : 0.7312  ██████████████
  akiec : 0.6337  ████████████
  bkl   : 0.6635  █████████████
  df    : 0.5882  ███████████
  vasc  : 0.8276  ████████████████

Confusion Matrix:
[[ 67  31   0   7   7   0   0]
 [ 84 542   8   6  13   1   1]
 [  1   4  34   2   2   0   1]
 [  0   3   2  32   4   0   0]
 [ 17  11   3  12  69   1   0]
 [  1   1   1   1   0   5   0]
 [  1   0   1   0   0   1  12]]

Classification Report:
   

In [1]:
import os
import getpass
import shutil

# ============================================================
# Setup: Repo + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Dataset Linking
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}"
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

print("\nUpdating repository...")
os.system("git pull origin main")

# ============================================================
# Download Git LFS checkpoints
# ============================================================

print("\nDownloading Git LFS checkpoints...")
os.system("git lfs install")
os.system("git lfs pull")

# ============================================================
# Experiment
# ============================================================

exp_name = "resnet50_v12recipe"

config_path = f"{REPO}/ham10000/configs/{exp_name}.yaml"
checkpoint_path = f"{REPO}/ham10000/checkpoints/{exp_name}/best_model.pt"

print("=" * 60)
print(f"RUNNING VALIDATION EVALUATION: {exp_name}")
print("=" * 60)

print(f"Config     : {config_path}")
print(f"Checkpoint : {checkpoint_path}")
print("Split      : val")

assert os.path.exists(config_path), f"Missing config: {config_path}"
assert os.path.exists(checkpoint_path), f"Missing checkpoint: {checkpoint_path}"

# ============================================================
# Evaluate on Validation Seta
# ============================================================

get_ipython().system(f"""
python {REPO}/ham10000/src/evaluate_test.py \
    --config {config_path} \
    --checkpoint {checkpoint_path} \
    --split val
""")

print("\nValidation evaluation completed.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...
Updating files: 100% (271/271), done.
Filtering content: 100% (4/4), 795.41 MiB | 7.44 MiB/s, done.
Encountered 6 file(s) that should have been pointers, but weren't:
	ham10000/checkpoints/baseline_image_only/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/last_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/best_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/last_model.pt
	ham10000/checkpoints/metadata_fusion/best_model.pt


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········



Updating repository...


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


Already up to date.

Updated git hooks.
Git LFS initialized.
RUNNING VALIDATION EVALUATION: resnet50_v12recipe
Config     : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/configs/resnet50_v12recipe.yaml
Checkpoint : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/resnet50_v12recipe/best_model.pt
Split      : val

Experiment  : resnet50_v12recipe
Checkpoint  : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/resnet50_v12recipe/best_model.pt
Split       : val
Metadata    : False
Device      : cuda

[val] 1009 images | mode=val
Loaded 1009 images for split='val'
Checkpoint loaded  (epoch 14, val_bal_acc=0.8010)

Running inference...

RESULTS  —  resnet50_v12recipe  —  split=val
Balanced Accuracy : 0.7746
Macro F1          : 0.6977
Macro ROC-AUC     : 0.9601

Per-class F1:
  mel   : 0.5145  ██████████
  nv    : 0.8376  ████████████████
  bcc   : 0.7692  ███████████████
  akiec : 0.6923  █████████████
  bkl   : 0.6120  

In [1]:
import os
import getpass
import shutil

# ============================================================
# Setup: Repo + Dataset Linking (Safe to Re-run)
# ============================================================

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"

if not os.path.exists(REPO):
    os.chdir("/kaggle/working")
    os.system(
        "git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git"
    )

os.chdir(REPO)

# ============================================================
# Dataset Linking
# ============================================================

KAGGLE_DATA = "/kaggle/input/datasets/rehmandaket/ham10000/data"
WORK = f"{REPO}/ham10000/data"

if not os.path.exists(KAGGLE_DATA):
    raise FileNotFoundError(
        f"KAGGLE_DATA not found at {KAGGLE_DATA}"
    )

os.makedirs(WORK, exist_ok=True)

print("Linking dataset...")

for item in os.listdir(KAGGLE_DATA):
    src = os.path.join(KAGGLE_DATA, item)
    dst = os.path.join(WORK, item)

    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.isdir(dst) and not os.path.islink(dst):
            shutil.rmtree(dst)
        else:
            os.remove(dst)

    os.symlink(src, dst)

print("Dataset linked successfully.")

# ============================================================
# Git Authentication
# ============================================================

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")

os.system('git config user.email "your_email@gmail.com"')
os.system('git config user.name "Ansaree"')

os.system(
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git'
)

print("\nUpdating repository...")
os.system("git pull origin main")

# ============================================================
# Download Git LFS Checkpoints
# ============================================================

print("\nDownloading Git LFS checkpoints...")
os.system("git lfs install")
os.system("git lfs pull")

# ============================================================
# Experiment
# ============================================================

exp_name = "resnet50_v12recipe"

config_path = f"{REPO}/ham10000/configs/{exp_name}.yaml"
checkpoint_path = f"{REPO}/ham10000/checkpoints/{exp_name}/best_model.pt"

assert os.path.exists(config_path), f"Missing config: {config_path}"
assert os.path.exists(checkpoint_path), f"Missing checkpoint: {checkpoint_path}"

# ============================================================
# Validation Evaluation
# ============================================================

print("=" * 60)
print(f"RUNNING VALIDATION EVALUATION: {exp_name}")
print("=" * 60)

get_ipython().system(f"""
python {REPO}/ham10000/src/evaluate_test.py \
    --config {config_path} \
    --checkpoint {checkpoint_path} \
    --split val
""")

# Preserve validation probabilities
val_src = f"{REPO}/ham10000/results/{exp_name}_probs.npz"
val_dst = f"{REPO}/ham10000/results/{exp_name}_val_probs.npz"

if os.path.exists(val_src):
    shutil.copy2(val_src, val_dst)
    print(f"Saved validation probabilities -> {val_dst}")

# ============================================================
# Test Evaluation
# ============================================================

print("\n" + "=" * 60)
print(f"RUNNING TEST EVALUATION: {exp_name}")
print("=" * 60)

get_ipython().system(f"""
python {REPO}/ham10000/src/evaluate_test.py \
    --config {config_path} \
    --checkpoint {checkpoint_path} \
    --split test
""")

# Preserve test probabilities
test_src = f"{REPO}/ham10000/results/{exp_name}_probs.npz"
test_dst = f"{REPO}/ham10000/results/{exp_name}_test_probs.npz"

if os.path.exists(test_src):
    shutil.copy2(test_src, test_dst)
    print(f"Saved test probabilities -> {test_dst}")

# ============================================================
# Calibration
# ============================================================

print("\n" + "=" * 60)
print(f"RUNNING CALIBRATION: {exp_name}")
print("=" * 60)

calibration_json = f"{REPO}/ham10000/results/{exp_name}_calibration.json"

get_ipython().system(f"""
python {REPO}/ham10000/src/calibrate.py \
    --val_probs {val_dst} \
    --test_probs {test_dst} \
    --out {calibration_json}
""")

print("\nCalibration completed.")

# ============================================================
# Commit & Push Results
# ============================================================

print("\nCommitting results to GitHub...")

os.system("git add ham10000/results/")
os.system('git commit -m "Add resnet50_v12recipe val/test probs + calibration results"')
os.system("git push origin main")

print("\nAll steps completed successfully.")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...
Updating files: 100% (275/275), done.
Filtering content: 100% (4/4), 795.41 MiB | 50.76 MiB/s, done.
Encountered 6 file(s) that should have been pointers, but weren't:
	ham10000/checkpoints/baseline_image_only/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/best_model.pt
	ham10000/checkpoints/densenet121_v12recipe/last_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/best_model.pt
	ham10000/checkpoints/efficientnet_b0_v12recipe/last_model.pt
	ham10000/checkpoints/metadata_fusion/best_model.pt


Linking dataset...
Dataset linked successfully.


GitHub Personal Access Token:  ········



Updating repository...
Already up to date.

Updated git hooks.
Git LFS initialized.


From https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging
 * branch            main       -> FETCH_HEAD


RUNNING VALIDATION EVALUATION: resnet50_v12recipe

Experiment  : resnet50_v12recipe
Checkpoint  : /kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/checkpoints/resnet50_v12recipe/best_model.pt
Split       : val
Metadata    : False
Device      : cuda

[val] 1009 images | mode=val
Loaded 1009 images for split='val'
Checkpoint loaded  (epoch 14, val_bal_acc=0.8010)

Running inference...

RESULTS  —  resnet50_v12recipe  —  split=val
Balanced Accuracy : 0.7746
Macro F1          : 0.6977
Macro ROC-AUC     : 0.9601

Per-class F1:
  mel   : 0.5145  ██████████
  nv    : 0.8376  ████████████████
  bcc   : 0.7692  ███████████████
  akiec : 0.6923  █████████████
  bkl   : 0.6120  ████████████
  df    : 0.6842  █████████████
  vasc  : 0.7742  ███████████████

Confusion Matrix:
[[ 89  10   1   2   7   2   0]
 [136 513   8   4  18   0   4]
 [  0   4  45   5   3   0   1]
 [  0   0   0  27   1   1   0]
 [  8  14   5  10  56   1   1]
 [  2   1   0   1   3  13   0]
 [  0   0   0   0   0   1

Everything up-to-date


## checkpoints need for ensembling

In [2]:

!git add .

In [3]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [7]:
!git commit -m "Add ResNet50 v12recipe evaluation results"

[main cf53c2f] Add ResNet50 v12recipe evaluation results
 5 files changed, 361 insertions(+), 28 deletions(-)
 create mode 100644 ham10000/results/resnet50_v12recipe_calibration.json
 rewrite ham10000/results/resnet50_v12recipe_report.txt (72%)
 create mode 100644 ham10000/results/resnet50_v12recipe_test_probs.npz
 create mode 100644 ham10000/results/resnet50_v12recipe_val_probs.npz


In [8]:
!git push origin main

Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 4 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 1.62 KiB | 1.62 MiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   7d1fc10..cf53c2f  main -> main


In [8]:
!git add ham10000/results/resnet18_v6_ema_sampler.json
!git add ham10000/results/resnet18_v6_ema_sampler_report.txt
!git add ham10000/results/resnet50_v12recipe.json
!git add ham10000/results/resnet50_v12recipe_report.txt

In [9]:
!git add ham10000/results/densenet121_v12recipe.json
!git add ham10000/results/densenet121_v12recipe_report.txt
!git add ham10000/results/densenet121_v12recipe_tta8.json
!git add ham10000/results/densenet121_v12recipe_tta8_report.txt

In [10]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   ham10000/results/densenet121_v12recipe.json
	new file:   ham10000/results/densenet121_v12recipe_report.txt
	new file:   ham10000/results/densenet121_v12recipe_tta8.json
	new file:   ham10000/results/densenet121_v12recipe_tta8_report.txt
	new file:   ham10000/results/resnet18_v6_ema_sampler.json
	new file:   ham10000/results/resnet18_v6_ema_sampler_report.txt
	new file:   ham10000/results/resnet50_v12recipe.json
	new file:   ham10000/results/resnet50_v12recipe_report.txt



In [11]:
!git commit -m "Add evaluation result files"

[main 4f94d3f] Add evaluation result files
 8 files changed, 135 insertions(+)
 create mode 100644 ham10000/results/densenet121_v12recipe.json
 create mode 100644 ham10000/results/densenet121_v12recipe_report.txt
 create mode 100644 ham10000/results/densenet121_v12recipe_tta8.json
 create mode 100644 ham10000/results/densenet121_v12recipe_tta8_report.txt
 create mode 100644 ham10000/results/resnet18_v6_ema_sampler.json
 create mode 100644 ham10000/results/resnet18_v6_ema_sampler_report.txt
 create mode 100644 ham10000/results/resnet50_v12recipe.json
 create mode 100644 ham10000/results/resnet50_v12recipe_report.txt


In [12]:
import os
import getpass

token = getpass.getpass("GitHub PAT: ")

os.system(
    f'git push https://{token}@github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git main'
)

GitHub PAT:  ········


To https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git
   188a352..4f94d3f  main -> main


0